# ABQ business data


## Setup

In [ ]:
%%capture
%%bash
pip install -U faker xlsxwriter python-magic
apt-get update
apt-get install -y tree zip unzip


In [ ]:
import os
import magic
import random
import duckdb
import pandas as pd
from pathlib import Path


In [ ]:
random.seed( 11 )

## Read in Excel files



### Fetch Excel files

In [ ]:
excel_zip_url = "https://ddc-datascience.s3.us-west-1.amazonaws.com/a-z.business/2023-08-21/excel_files.zip"
excel_zip_url

'https://ddc-datascience.s3.us-west-1.amazonaws.com/a-z.business/2023-08-21/excel_files.zip'

In [ ]:
cd /content/


/content


In [ ]:
!rm -rf xls
!mkdir xls

In [ ]:
cd xls

/content/xls


In [ ]:
!curl -O {excel_zip_url}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  118M  100  118M    0     0  41.8M      0  0:00:02  0:00:02 --:--:-- 41.8M


In [ ]:
!ls -l --si

total 125M
-rw-r--r-- 1 root root 125M Feb 21 04:24 excel_files.zip


In [ ]:
!unzip -q excel_files.zip


In [ ]:
!find a-z.abq.business/ -type f | wc -l


12704


### Traverse nested folders



#### Create a list of all files

In [ ]:
files_list = [
  str(file)
    for file in Path("./a-z.abq.business/").rglob('*')
      if file.is_file()
]
files_list[:10]

['a-z.abq.business/file00151.xlsx',
 'a-z.abq.business/file05669.xlsx',
 'a-z.abq.business/file10412.xlsx',
 'a-z.abq.business/file07352.xlsx',
 'a-z.abq.business/file07936.xlsx',
 'a-z.abq.business/file07894.xlsx',
 'a-z.abq.business/file09399.xlsx',
 'a-z.abq.business/file02099.xlsx',
 'a-z.abq.business/file05299.xlsx',
 'a-z.abq.business/file09992.xlsx']

#### Filter for Excel files

In [ ]:
file_magic = magic.Magic()
excel_files = [
  file
    for file in files_list
      for file_type in [ file_magic.from_file(file) ]
        if 'Microsoft Excel' in file_type
]
random.shuffle( excel_files )

In [ ]:
len( excel_files )

12704

In [ ]:
excel_files[:10]

['a-z.abq.business/focus/music/issue/force/file05725.xlsx',
 'a-z.abq.business/state/file12416.xlsx',
 'a-z.abq.business/maybe/after/spend/quite/later/level/south/file12032.xlsx',
 'a-z.abq.business/sport/catch/stuff/begin/those/sense/file05435.xlsx',
 'a-z.abq.business/first/piece/thing/score/stage/table/while/file08167.xlsx',
 'a-z.abq.business/file00868.xlsx',
 'a-z.abq.business/focus/music/issue/force/file06704.xlsx',
 'a-z.abq.business/force/phone/three/file10492.xlsx',
 'a-z.abq.business/force/phone/three/front/file08988.xlsx',
 'a-z.abq.business/hotel/first/coach/file07959.xlsx']

### Read Excel files


Change start and/or end to slice the list of Excel files.

In [ ]:
start = 0  # change to None to start at the beginning
stop = 10  # change to None to stop at the end
excel_dfs = [
  pd.read_excel( file_path, sheet_name='Data', dtype=str )
    for file_path in excel_files[ start : stop ]
]
len( excel_dfs )

10

In [ ]:
excel_dfs[0]

,Unnamed: 0,Source,Date,Obsolescence Date,Business Name,Legal Name,Physical Address,Physical Address Number,Physical Pre Direction,Physical Address Name,...,Est. Rent Annual Expense,Est. Technology Annual Expense,Est. Telecom Annual Expense,Est. Utilities Annual Expense,AtoZ ID,Home Based Business,Franchise Type,Holding Parent Name,Source,Unnamed: 227
0,17175,AtoZDatabases,08/21/2023,02/21/2024,Indira D Veerapaneni MD,Pediatric Intensive Care,1100 Central Ave SE,1100,NaN,Central,...,Professional,Professional,Professional,Professional,11140015909579,No,NaN,NaN,AtoZDatabases,NaN
1,17176,AtoZDatabases,08/21/2023,02/21/2024,Indoor Flea Market,NaN,6201 Central Ave NE,6201,NaN,Central,...,"$5,000 to $9,999","$5,000 to $9,999","Up to $1,999","$7,500 to $19,999",11140023881492,No,NaN,NaN,AtoZDatabases,NaN
2,17177,AtoZDatabases,08/21/2023,02/21/2024,Indoor Jtown,NaN,1720 Bridge Blvd SW Ste 103,1720,NaN,Bridge,...,"$10,000 to $24,999","$10,000 to $24,999","Up to $1,999","$7,500 to $19,999",11140035794274,No,NaN,NaN,AtoZDatabases,NaN


## Create a dataframe

### Concatenate and transform collection of data frames

In [ ]:
result_df = (
    pd.concat( excel_dfs )
    .rename( columns = { 'Unnamed: 0': 'Index' } )
    .assign( Index = lambda x: x['Index'].astype( int ) )
    .sort_values( by=["Index"] )
    .set_index( 'Index' )
)
result_df.shape

(30, 228)

In [ ]:
result_df[:10]

,Source,Date,Obsolescence Date,Business Name,Legal Name,Physical Address,Physical Address Number,Physical Pre Direction,Physical Address Name,Physical Address Suffix,...,Est. Rent Annual Expense,Est. Technology Annual Expense,Est. Telecom Annual Expense,Est. Utilities Annual Expense,AtoZ ID,Home Based Business,Franchise Type,Holding Parent Name,Source,Unnamed: 227
Index,,,,,,,,,,,,,,,,,,,,,
2604,AtoZDatabases,08/10/2023,02/10/2024,Always Deals For Days,NaN,6115 Murcia Ave NW,6115,NaN,Murcia,Ave,...,"$100,000 or more","$100,000 or more","$25,000 to $99,999","$100,000 or more",11140004571839,No,NaN,NaN,AtoZDatabases,NaN
2605,AtoZDatabases,08/10/2023,02/10/2024,Always Fire Inc,NaN,3800 Hawkins St NE Ste D,3800,NaN,Hawkins,St,...,"$2,500 to $4,999","$2,500 to $4,999","Up to $1,999","$2,500 to $7,499",11132545303577,No,NaN,NaN,AtoZDatabases,NaN
2606,AtoZDatabases,08/10/2023,02/10/2024,Always Green,NaN,8016 Bluffs Edge Pl NW,8016,NaN,Bluffs Edg,Pl,...,"$5,000 to $9,999","$5,000 to $9,999","$2,000 to $4,999","$2,500 to $7,499",11132553966795,Yes,NaN,NaN,AtoZDatabases,NaN
16305,AtoZDatabases,08/10/2023,02/10/2024,Herring Jennifer Rn,NaN,6400 Uptown Blvd NE,6400,NaN,Uptown,Blvd,...,"$25,000 to $99,999","$10,000 to $24,999","$10,000 to $24,999","$7,500 to $19,999",11140016834376,No,NaN,NaN,AtoZDatabases,NaN
16306,AtoZDatabases,08/10/2023,02/10/2024,Hers & His Gift & Collectibles,NaN,6001 Winter Haven Dr NW,6001,NaN,Winter Hav,Dr,...,"$2,500 to $4,999","$2,500 to $4,999","Up to $1,999","$2,500 to $7,499",11134004971726,No,NaN,NaN,AtoZDatabases,NaN
16307,AtoZDatabases,08/10/2023,02/10/2024,Hertz,NaN,3400 University Blvd SE Ste I,3400,NaN,University,Blvd,...,"$5,000 to $9,999","$5,000 to $9,999","$2,000 to $4,999","$2,500 to $7,499",11132544449370,No,True Franchise,Hertz Global Holdings Inc,AtoZDatabases,NaN
17175,AtoZDatabases,08/21/2023,02/21/2024,Indira D Veerapaneni MD,Pediatric Intensive Care,1100 Central Ave SE,1100,NaN,Central,Ave,...,Professional,Professional,Professional,Professional,11140015909579,No,NaN,NaN,AtoZDatabases,NaN
17176,AtoZDatabases,08/21/2023,02/21/2024,Indoor Flea Market,NaN,6201 Central Ave NE,6201,NaN,Central,Ave,...,"$5,000 to $9,999","$5,000 to $9,999","Up to $1,999","$7,500 to $19,999",11140023881492,No,NaN,NaN,AtoZDatabases,NaN
17177,AtoZDatabases,08/21/2023,02/21/2024,Indoor Jtown,NaN,1720 Bridge Blvd SW Ste 103,1720,NaN,Bridge,Blvd,...,"$10,000 to $24,999","$10,000 to $24,999","Up to $1,999","$7,500 to $19,999",11140035794274,No,NaN,NaN,AtoZDatabases,NaN


### Save data frame to parquet file

In [ ]:
result_df.to_parquet( "a-z.business.parquet" )

## Create a Database

In [ ]:
# Path to your Parquet file
parquet_file_path = 'a-z.business.parquet'
parquet_file_path

'a-z.business.parquet'

In [ ]:
# Create a DuckDB connection
conn = duckdb.connect()


In [ ]:
# Create query for the number of rows in the Parquet file
query = f'''
  SELECT
    COUNT(1) AS row_count
  FROM
    '{parquet_file_path}'
'''
print( query )



  SELECT
    COUNT(1) AS row_count
  FROM
    'a-z.business.parquet'



In [ ]:
# Run query
result_df = conn.execute(query).fetchdf()
result_df

,row_count
0,30
